# Stage 2 — Consideration: Campaign Intelligence Crew Walkthrough

**Book**: *Mastering Agentic AI for Customer Journey Marketing*  
**Author**: Pushparajan Ramar  
**Chapters**: 5–6  
**Framework**: CrewAI  
**Repo**: https://github.com/Pushparajan/agenticai-marketing

---

This notebook walks through the Campaign Intelligence Crew step by step:

1. **Setup** — Install dependencies and configure environment
2. **Tools** — Explore each tool individually with mock data
3. **Agents** — Inspect the three crew agents and their configurations
4. **Crew Run** — Execute the full crew pipeline
5. **Flow** — Run the Consideration Journey Flow end-to-end
6. **Analysis** — Examine and visualise the outputs

## 1. Setup

Make sure you are running from the repository root or have the repo on your Python path.

In [ ]:
# Ensure repo root is on the path
import sys
from pathlib import Path

repo_root = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Set mock mode (no API keys required)
import os
os.environ["USE_MOCK"] = "true"

print(f"Repo root: {repo_root}")
print(f"USE_MOCK: {os.environ['USE_MOCK']}")

In [ ]:
# Install dependencies if needed (uncomment in Colab/Binder)
# !pip install crewai pydantic pyyaml httpx python-dotenv rich

## 2. Explore the Tools

Each tool follows the **USE_MOCK** pattern: when `USE_MOCK=true`, it returns
realistic marketing data without calling any external API. Let's try each one.

### 2.1 Market Research Tools

In [ ]:
import json

from stage2_consideration.tools.market_research_tools import (
    web_search,
    linkedin_company_lookup,
    get_industry_benchmarks,
    get_tech_stack_signals,
)

# Web search
result = web_search.run("Acme Corp B2B SaaS")
print("=== Web Search ===")
print(json.dumps(json.loads(result), indent=2))

In [ ]:
# LinkedIn company lookup
result = linkedin_company_lookup.run("Acme Corp")
print("=== LinkedIn Company Lookup ===")
print(json.dumps(json.loads(result), indent=2))

In [ ]:
# Industry benchmarks
result = get_industry_benchmarks.run("B2B SaaS")
print("=== Industry Benchmarks ===")
print(json.dumps(json.loads(result), indent=2))

In [ ]:
# Tech stack signals
result = get_tech_stack_signals.run("acmecorp.com")
print("=== Tech Stack Signals ===")
print(json.dumps(json.loads(result), indent=2))

### 2.2 Competitor Intelligence Tools

In [ ]:
from stage2_consideration.tools.competitor_intel_tools import (
    get_competitor_positioning,
    search_competitor_campaigns,
)

# Competitor positioning
result = get_competitor_positioning.run("RivalTech")
print("=== Competitor Positioning ===")
print(json.dumps(json.loads(result), indent=2))

In [ ]:
# Competitor campaigns
result = search_competitor_campaigns.run(industry="B2B SaaS", channel="email")
print("=== Competitor Campaigns (email) ===")
print(json.dumps(json.loads(result), indent=2))

### 2.3 Segment CDP Tools

In [ ]:
from stage2_consideration.tools.segment_cdp_tools import (
    query_segment_cdp,
    get_contact_journey_events,
)

# Query audience segment
result = query_segment_cdp.run("high_intent_mql")
print("=== Segment CDP: Audience ===")
print(json.dumps(json.loads(result), indent=2))

In [ ]:
# Contact journey events
result = get_contact_journey_events.run("alex.rivera@acmecorp.com")
parsed = json.loads(result)
print("=== Journey Events ===")
print(f"Total events: {parsed['total_events']}")
print(f"Current stage: {parsed['journey_summary']['current_stage']}")
print(f"\nEvent timeline:")
for evt in parsed['events']:
    print(f"  {evt['timestamp'][:10]}  {evt['event']:25s}  {evt.get('page', evt.get('campaign', evt.get('asset', evt.get('webinar', ''))))}}")

### 2.4 Content Publisher Tools

In [ ]:
from stage2_consideration.tools.content_publisher_tools import (
    publish_to_notion,
    get_content_library,
    get_case_studies,
)

# Content library
result = get_content_library.run("case_study")
parsed = json.loads(result)
print(f"=== Content Library ({parsed['total_assets']} assets) ===")
for asset in parsed['assets']:
    print(f"  [{asset['id']}] {asset['title']}")

In [ ]:
# Case studies
result = get_case_studies.run("B2B SaaS")
parsed = json.loads(result)
print(f"=== Case Studies ({parsed['total_results']} found) ===")
for cs in parsed['case_studies']:
    print(f"\n  {cs['title']}")
    print(f"  Customer: {cs['customer']} ({cs['company_size']})")
    print(f"  Key result: {cs['results']}")

In [ ]:
# Publish to Notion (mock)
result = publish_to_notion.run(
    title="Acme Corp Nurture Email #1",
    content="Subject: Unlock 38% lower CAC — here's how...\n\nHi Alex, ..."
)
print("=== Notion Publish ===")
print(json.dumps(json.loads(result), indent=2))

## 3. Inspect Agent Configuration

The crew agents are defined in YAML and loaded by the `@CrewBase` decorator.

In [ ]:
import yaml

config_dir = repo_root / "stage2_consideration" / "crews" / "campaign_intelligence_crew" / "config"

# Load agents config
with open(config_dir / "agents.yaml") as f:
    agents_cfg = yaml.safe_load(f)

for name, cfg in agents_cfg.items():
    print(f"\n{'=' * 50}")
    print(f"Agent: {name}")
    print(f"Role : {cfg['role']}")
    print(f"Goal : {cfg['goal'][:120]}...")
    print(f"{'=' * 50}")

In [ ]:
# Load tasks config
with open(config_dir / "tasks.yaml") as f:
    tasks_cfg = yaml.safe_load(f)

for name, cfg in tasks_cfg.items():
    print(f"\n{'=' * 50}")
    print(f"Task   : {name}")
    print(f"Agent  : {cfg['agent']}")
    context = cfg.get('context', [])
    if context:
        print(f"Context: {context}")
    print(f"Description: {cfg['description'][:150]}...")
    print(f"{'=' * 50}")

## 4. Run the Full Crew

Now let's run the complete Campaign Intelligence Crew. In mock mode this
will simulate all API calls and produce a full campaign package.

> **Note**: This cell requires an `OPENAI_API_KEY` (or Ollama running
> locally) since CrewAI uses an LLM to orchestrate agent reasoning.

In [ ]:
# Uncomment one of these options:

# Option A — Use OpenAI
# os.environ["OPENAI_API_KEY"] = "sk-..."

# Option B — Use Ollama (free, local)
# os.environ["OPENAI_API_KEY"] = "ollama"
# os.environ["OPENAI_BASE_URL"] = "http://localhost:11434/v1"

print("Configure your LLM above, then run the next cell.")

In [ ]:
from stage2_consideration.crews.campaign_intelligence_crew.crew import (
    CampaignIntelligenceCrew,
)

inputs = {
    "company": "Acme Corp",
    "industry": "B2B SaaS",
    "channels": "email,linkedin,webinar",
}

crew_instance = CampaignIntelligenceCrew()
result = crew_instance.crew().kickoff(inputs=inputs)

print("\n" + "=" * 60)
print("CREW RESULT")
print("=" * 60)
print(result.raw if hasattr(result, 'raw') else str(result))

## 5. Run the Consideration Journey Flow

The flow wraps the crew in a multi-step pipeline that also publishes
results to Klaviyo and HubSpot (mocked in demo mode).

In [ ]:
from stage2_consideration.flows.consideration_journey_flow import (
    ConsiderationJourneyFlow,
)

flow = ConsiderationJourneyFlow()
flow.state.mql_email = "alex.rivera@acmecorp.com"
flow.state.company = "Acme Corp"
flow.state.industry = "B2B SaaS"
flow.state.channels = "email,linkedin,webinar"

flow_result = flow.kickoff()

print("\n" + "=" * 60)
print("FLOW STATE")
print("=" * 60)
print(f"  Klaviyo  : {flow.state.klaviyo_status}")
print(f"  HubSpot  : {flow.state.hubspot_status}")
print(f"  Fallback : {flow.state.fallback_used}")
print(f"  Error    : {flow.state.error or 'None'}")

## 6. Analyse Outputs

Let's load and examine the saved crew output file.

In [ ]:
output_path = (
    repo_root / "stage2_consideration" / "crews"
    / "campaign_intelligence_crew" / "output" / "campaign_output.json"
)

if output_path.exists():
    with open(output_path) as f:
        output_data = json.load(f)
    print("Output file loaded successfully.")
    print(f"Keys: {list(output_data.keys())}")
    if "metadata" in output_data:
        print(f"\nMetadata:")
        print(json.dumps(output_data['metadata'], indent=2))
else:
    print(f"Output file not found at {output_path}")
    print("Run the crew first (Section 4) to generate output.")

---

## Summary

In this walkthrough you have:

1. **Explored** each tool individually and seen the mock data it returns
2. **Inspected** the YAML configuration for agents and tasks
3. **Executed** the Campaign Intelligence Crew end-to-end
4. **Run** the Consideration Journey Flow with Klaviyo + HubSpot activation
5. **Analysed** the structured output

### Next steps

- Set `USE_MOCK=false` and provide real API keys to run against live services
- Customise the agents in `config/agents.yaml` for your brand voice
- Add new tools (e.g. a G2 review scraper) to enrich the research step
- Proceed to **Stage 3 — Decision** (LangGraph) in `stage3_decision/`

---

*Companion code for "Mastering Agentic AI for Customer Journey Marketing" by Pushparajan Ramar*